# Stylized facts (1–3) for BTC-EUR realized measures

**Goal.** Document distributional properties, volatility clustering, and long
memory of BTC-EUR daily returns and realized variance, and translate each
finding into a constraint on the HAR-CJ-X-Q vs HAR-RS-X-Q model family.

**Data.** `realized_measures_btc_eur_15m.parquet` — 316 days of daily realized
measures computed from 15-minute Bitvavo candles (M ≈ 96 bars/day).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t as student_t

from hartrace.stylized import (
    moments, fit_student_t, fit_hansen_skewt, qq_points,
    hill_estimator, gpd_tail,
    acf_with_ci, ljung_box, arch_lm,
    hurst_rs, gph,
)

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

df = pd.read_parquet(Path.cwd().parent / 'data/processed/realized_measures_btc_eur_15m.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['log_RV'] = np.log(df['RV'])
df['r2'] = df['r_d'] ** 2
df['abs_r'] = np.abs(df['r_d'])

print(f"N = {len(df)} days, {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

## Fact 1 — Distributional properties

The conditioning distribution dictates the innovation choice in the HAR model.
Three layers:

1. **Daily log-returns** $r_t$ — determines the η-distribution in
   $r_{t+1} = \mu_r + \sqrt{RV_{t+1}}\cdot \eta_{t+1}$.
2. **log realized variance** $\log RV_t$ — determines the ε-distribution in
   $\log RV_{t+1} = \mu_t + \sigma_t \varepsilon_{t+1}$.
3. **Tail behavior** of |returns| — quantifies how heavy the heavy tails are.

### 1a. Daily log-returns


In [ ]:
r = df['r_d'].values
mom_r = moments(r)
t_r = fit_student_t(r)
st_r = fit_hansen_skewt(r)

print("Moments of daily log-returns r_d")
for k, v in mom_r.items():
    print(f"  {k:>12}: {v:+.6g}")
print(f"\nSymmetric Student-t MLE:  ν̂ = {t_r['nu']:.2f}, μ̂ = {t_r['mu']:+.5f}, σ̂ = {t_r['sigma']:.5f}")
print(f"Hansen skewed-t MLE   :  ν̂ = {st_r['nu']:.2f}, λ̂ = {st_r['lam']:+.3f}, log-lik = {st_r['loglik']:.2f}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 8))
ax[0, 0].plot(df['date'], r, lw=0.7, color='#2b6cb0'); ax[0, 0].axhline(0, color='k', lw=0.4)
ax[0, 0].set_title('$r_t$ over time'); ax[0, 0].set_ylabel('$r_t$')

xs = np.linspace(r.min(), r.max(), 400)
ax[0, 1].hist(r, bins=50, density=True, alpha=0.45, color='#a0aec0', edgecolor='white')
ax[0, 1].plot(xs, norm.pdf(xs, mom_r['mean'], mom_r['std']), color='#2b6cb0', label='Normal')
ax[0, 1].plot(xs, student_t.pdf(xs, df=t_r['nu'], loc=t_r['mu'], scale=t_r['sigma']),
              color='#d97706', label=f"Student-t (ν̂={t_r['nu']:.1f})")
ax[0, 1].legend(); ax[0, 1].set_title('Histogram + fitted densities')

th, sa = qq_points(r, dist='norm')
ax[1, 0].scatter(th, sa, s=10, color='#2b6cb0', alpha=0.6)
lim = [min(th.min(), sa.min()), max(th.max(), sa.max())]
ax[1, 0].plot(lim, lim, 'k--', lw=0.7); ax[1, 0].set_title('QQ vs Gaussian')

th2, sa2 = qq_points(r, dist='skewt', params=st_r)
ax[1, 1].scatter(th2, sa2, s=10, color='#d97706', alpha=0.6)
lim2 = [min(th2.min(), sa2.min()), max(th2.max(), sa2.max())]
ax[1, 1].plot(lim2, lim2, 'k--', lw=0.7); ax[1, 1].set_title(f'QQ vs Hansen skew-t (ν̂={st_r["nu"]:.1f}, λ̂={st_r["lam"]:+.2f})')
plt.tight_layout(); plt.show()

**Interpretation.** Excess kurtosis on returns is large (~2.5; Gaussian = 0). The
Student-t fit pushes the degrees of freedom right down to **ν ≈ 3–4** — these are
genuinely heavy-tailed. Even skewed-t needs ν ≈ 4 to fit. The asymmetry parameter
λ is small and slightly negative — minor downside emphasis but symmetric for
practical purposes.

**Model implication.** For the *return innovation* η in
$r_{t+1} = \sqrt{RV_{t+1}}\cdot\eta_{t+1}$, skew-t with ν ≈ 4 is essential.
A Gaussian innovation would catastrophically underestimate tail risk for crypto.

### 1b. log realized variance — the actual modelling target


In [ ]:
lrv = df['log_RV'].values
mom_lrv = moments(lrv)
t_lrv = fit_student_t(lrv)
st_lrv = fit_hansen_skewt(lrv)

print(f"Moments of log RV")
for k, v in mom_lrv.items():
    print(f"  {k:>12}: {v:+.6g}")
print(f"\nSymmetric Student-t MLE:  ν̂ = {t_lrv['nu']:.2f}")
print(f"Hansen skewed-t MLE   :  ν̂ = {st_lrv['nu']:.2f}, λ̂ = {st_lrv['lam']:+.3f}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 8))
ax[0, 0].plot(df['date'], lrv, lw=0.7, color='#7c3aed')
ax[0, 0].set_title('$\\log RV_t$ over time'); ax[0, 0].set_ylabel('$\\log RV_t$')

xs = np.linspace(lrv.min(), lrv.max(), 400)
ax[0, 1].hist(lrv, bins=50, density=True, alpha=0.45, color='#a0aec0', edgecolor='white')
ax[0, 1].plot(xs, norm.pdf(xs, mom_lrv['mean'], mom_lrv['std']), color='#7c3aed', label='Normal')
ax[0, 1].legend(); ax[0, 1].set_title('Histogram + Normal density')

th, sa = qq_points(lrv, dist='norm')
ax[1, 0].scatter(th, sa, s=10, color='#7c3aed', alpha=0.6)
lim = [min(th.min(), sa.min()), max(th.max(), sa.max())]
ax[1, 0].plot(lim, lim, 'k--', lw=0.7); ax[1, 0].set_title('QQ vs Gaussian')

th2, sa2 = qq_points(lrv, dist='skewt', params=st_lrv)
ax[1, 1].scatter(th2, sa2, s=10, color='#d97706', alpha=0.6)
lim2 = [min(th2.min(), sa2.min()), max(th2.max(), sa2.max())]
ax[1, 1].plot(lim2, lim2, 'k--', lw=0.7); ax[1, 1].set_title(f'QQ vs skew-t (ν̂={st_lrv["nu"]:.1f}, λ̂={st_lrv["lam"]:+.2f})')
plt.tight_layout(); plt.show()

**Interpretation.** A very different picture from returns: excess kurtosis on log RV is
only ~0.3 — close to Gaussian. JB rejects normality only marginally (p ≈ 0.04). The
skewed-t fit gives ν ≈ 13 — significantly higher df than for returns. There *is* a
notable left-skew (λ ≈ −0.18): negative shocks to vol are more frequent than positive.

**Model implication.** The ε-innovation in the HAR mean equation can use a *milder*
skewed-t — ν ≈ 10–15 rather than 4. The skew parameter λ ≈ −0.2 is non-trivial and
justifies keeping the asymmetric distribution rather than collapsing to plain Student-t.

### 1c. Tail behavior — Hill estimator and GPD fit


In [ ]:
k_r, xi_r = hill_estimator(r, side='two-sided')
gpd_r = gpd_tail(r, quantile=0.95)
gpd_lrv = gpd_tail(lrv - lrv.mean(), quantile=0.95)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(k_r, xi_r, lw=1.2, color='#2b6cb0'); ax[0].axhline(0.5, color='r', linestyle='--', lw=0.8, label='ξ=1/2')
ax[0].set_title('Hill estimator on |r_d|'); ax[0].set_xlabel('k'); ax[0].set_ylabel('ξ̂(k)'); ax[0].legend()
ax[1].axis('off')
ax[1].text(0.0, 0.9, f"GPD fit above 95-pctile:\n\n"
                     f"|r_d|:  ξ̂ = {gpd_r['xi']:+.3f}, β̂ = {gpd_r['beta']:.5f}\n"
                     f"        n_exc = {gpd_r['n_exc']}\n\n"
                     f"|log RV - μ|: ξ̂ = {gpd_lrv['xi']:+.3f}, β̂ = {gpd_lrv['beta']:.3f}\n"
                     f"        n_exc = {gpd_lrv['n_exc']}",
            va='top', family='monospace', fontsize=10)
plt.tight_layout(); plt.show()

## Fact 2 — Volatility clustering / autocorrelation

The traditional ARCH/GARCH literature assumes vol clustering shows up in the
autocorrelation of squared/absolute returns. For low-frequency aggregated data
this can be very weak — the realized-variance literature emerged precisely to
recover the persistent vol signal that gets buried in noisy r².

We examine four series:

| Series | What it captures | If autocorrelated → |
|---|---|---|
| $r_t$ | Return predictability | Market inefficiency |
| $|r_t|$ | Vol clustering, low-frequency | ARCH-style models |
| $r_t^2$ | Vol clustering, low-frequency | ARCH-style models |
| $\log RV_t$ | Vol clustering, high-resolution | HAR/RV models |


In [ ]:
series = {'r_t': r, '|r_t|': np.abs(r), 'r_t^2': r**2, 'log RV': lrv}
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax_, (lab, x) in zip(axes.ravel(), series.items()):
    ac, ci = acf_with_ci(x, nlags=30)
    lags = np.arange(len(ac))
    cw = (ci[1, 1] - ci[1, 0]) / 2
    ax_.bar(lags, ac, width=0.6, color=['#2b6cb0' if i>0 else '#9ca3af' for i in lags])
    ax_.fill_between([0, 30], -cw, cw, color='red', alpha=0.10)
    ax_.axhline(0, color='k', lw=0.4)
    ax_.set_title(f'ACF of {lab}'); ax_.set_xlim(-0.5, 30.5)
plt.tight_layout(); plt.show()

In [ ]:
rows = []
for lab, x in [('r_d', r), ('|r_d|', np.abs(r)), ('r_d^2', r**2), ('log_RV', lrv)]:
    for lg in (5, 10, 22):
        lb = ljung_box(x, lags=[lg])
        rows.append({'series': lab, 'test': 'LB', 'lag': lg,
                     'stat': float(lb['lb_stat'].iloc[0]),
                     'pvalue': float(lb['lb_pvalue'].iloc[0])})
for lg in (5, 10, 22):
    lm = arch_lm(r, nlags=lg)
    rows.append({'series': 'r_d', 'test': 'ARCH-LM', 'lag': lg,
                 'stat': lm['lm_stat'], 'pvalue': lm['lm_pvalue']})
pd.DataFrame(rows).pivot_table(index=['series','test'], columns='lag', values='pvalue').round(4)

**Interpretation.**

- **Returns** $r_t$: no autocorrelation at any lag (LB p > 0.4). Returns are unpredictable.
- **$|r_t|$** and $r_t^2$: barely any clustering (LB p > 0.07). This is *unusual* for
  financial data — classical ARCH/GARCH machinery assumes strong clustering here.
- **log RV**: massive clustering (LB p ≈ 10⁻⁵² at lag 22). The persistent vol signal is
  clearly there, but only visible at the realized-variance level.
- **ARCH-LM on returns**: not significant (p > 0.12). A direct GARCH on returns
  would struggle to find structure.

**Model implication.** This is a strong methodological win for HAR-style models.
The traditional GARCH-on-returns approach is essentially blind to the vol structure
here, while HAR-on-realized-variance picks it up cleanly. This empirically validates
the entire HAR-CJ-X-Q / HAR-RS-X-Q framework choice over GARCH alternatives.

## Fact 3 — Long memory

**R/S analysis.** H = 0.5 is Brownian (no memory). H > 0.5 = positive long-range
dependence. Empirical equity vol typically shows H ≈ 0.75–0.85.

**GPH estimator.** d = 0 is short memory. d ∈ (0, 0.5) is stationary long memory.
Equity vol typically gives d ≈ 0.4.


In [ ]:
rs_r = hurst_rs(np.abs(r))
rs_lrv = hurst_rs(lrv)
gph_r = gph(np.abs(r))
gph_lrv = gph(lrv)

print(f"R/S Hurst on |r_d|  : H = {rs_r['H']:.3f}  (SE {rs_r['se']:.3f})")
print(f"R/S Hurst on log RV : H = {rs_lrv['H']:.3f}  (SE {rs_lrv['se']:.3f})")
print(f"GPH d  on |r_d|     : d = {gph_r['d']:+.3f}  (SE {gph_r['se']:.3f}, p = {gph_r['pvalue_two_sided']:.3g})")
print(f"GPH d  on log RV    : d = {gph_lrv['d']:+.3f}  (SE {gph_lrv['se']:.3f}, p = {gph_lrv['pvalue_two_sided']:.3g})")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
for label, rs, col in [('log RV', rs_lrv, '#7c3aed'), ('|r_d|', rs_r, '#2b6cb0')]:
    ax.scatter(rs['log_n'], rs['log_rs'], s=22, color=col, alpha=0.8, label=f"{label} (H={rs['H']:.3f})")
    ax.plot(rs['log_n'], rs['fit_intercept'] + rs['H'] * rs['log_n'], color=col, lw=1.2)
H0_line = 0.5 * rs_lrv['log_n']; H0_line = H0_line + (rs_lrv['log_rs'].mean() - H0_line.mean())
ax.plot(rs_lrv['log_n'], H0_line, 'k--', lw=0.8, label='H=1/2 (no memory)')
ax.set_xlabel('log n'); ax.set_ylabel('log E[R/S]'); ax.set_title('R/S analysis')
ax.legend(); plt.tight_layout(); plt.show()

**Interpretation.**

- **log RV: H = 0.875** (R/S) and **d = 0.453** (GPH, p = 0.004) → unambiguous, strong
  long memory in volatility. This is right at the stationarity boundary (d → 0.5).
- **|r_d|: H = 0.710** and d = 0.268 (p = 0.085) → also long memory, but weaker
  signal because daily returns are a noisier vol proxy.

These values are *in line with* canonical equity vol benchmarks. BTC-EUR
volatility persistence is comparable to S&P 500.

**Model implication.** The HAR cascade (D + W + M) is genuinely justified. The
daily lag β_D will dominate (consistent with ACF(1) ≈ 0.45), but β_W and β_M will
be significant because of the long-memory tail picked up by H = 0.87. Considering
fractional integration (FIGARCH-style) as a separate benchmark is reasonable since
d̂ ≈ 0.45 is borderline.

## Summary — model implications

| Fact | Finding | Effect on model |
|---|---|---|
| 1a. Return tails | ν̂ ≈ 3.8, λ̂ ≈ −0.07 | **Hansen skew-t mandatory** for return innovation η |
| 1b. log RV dist. | ν̂ ≈ 13, λ̂ ≈ −0.18 | Skew-t for ε but milder; Gaussian only marginally rejected |
| 2. Clustering | LB p ≈ 10⁻⁵² on log RV; not on r² | HAR > GARCH for crypto vol |
| 3. Long memory | H = 0.87, d = 0.45 (log RV) | W and M cascade components essential |

These results validate the architectural choices made in the modeling phase. The
next stylized-facts iteration (4–7) will probe jumps in time, asymmetry, periodicity,
and microstructure before parameter estimation.
